In [45]:
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
import os
import json
from pprint import pprint

@tool
def get_current_weather(location: str) -> str:
    """根据给定的地区和单位获取天气"""
    weather_info = {
        "location": location,
        "temperature": "32",
        "unit": '摄氏度',
        "forecast": ["晴天", "多云"],
    }
    return json.dumps(obj=weather_info, ensure_ascii=False)

tools = [get_current_weather]

print(get_current_weather.invoke(dict(location="北京")))

llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

llm_with_tools = llm.bind_tools(tools)

messages = [
    SystemMessage(content="关于天气的问题，请自行调用工具查询。"),
    HumanMessage(content="北京的天气怎么样？")
]

ai_msg = llm_with_tools.invoke(messages)

messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    selected_tool = {"get_current_weather": get_current_weather}[tool_call["name"]]
    tool_output = selected_tool.invoke(tool_call["args"])
    messages.append(ToolMessage(tool_output, tool_call_id=tool_call["id"]))

pprint(messages)

output = llm.invoke(messages)

output.content

{"location": "北京", "temperature": "32", "unit": "摄氏度", "forecast": ["晴天", "多云"]}
[SystemMessage(content='关于天气的问题，请自行调用工具查询。', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='北京的天气怎么样？', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_cdf9d174f83a409e880c35', 'function': {'arguments': '{"location": "北京"}', 'name': 'get_current_weather'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 169, 'total_tokens': 189, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'qwen-turbo', 'system_fingerprint': None, 'id': 'chatcmpl-cf059f1f-d757-92da-ab17-becc7c6949b5', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--fb580c35-9e67-4cc5-a92a-1bf9e9b2de05-0', tool_calls=[{'name': 'get_current_weather', 'args': {'location': '北京'}, 'id': 'ca

'北京的天气目前是32摄氏度，天气晴朗，偶尔会有云。'